# PS2: How to Make AI Applications Reduce Sycophancy
Yongzhuo Wu · FP1 · Session B · yw650@duke.edu  
COMSCI/ECON 206, Autumn 2026 Session 1 · Professor Luyao Zhang

This notebook reproduces the paper's hypothetical correction market and delivery-cost challenge. The retained behavioral record is one author self-reflection, with no observed purchases or population estimate.

**Run in Colab:** select **Runtime → Run all**. Python 3.12 or later is required. The first cell installs pinned dependencies into a dedicated environment and leaves Colab's preloaded NumPy/SciPy unchanged. Initial installation needs internet; the calculation needs no external dataset, API key, or random seed.

**Author execution:** I ran all eight code cells in Colab on October 6, 2026, at 11:08:10 UTC (19:08:10 Beijing time), using Python 3.13.16. The outputs below are from that run, including 14,406 bid-deviation checks, four matching Nashpy cases, and 121 cost-boundary mixed-strategy pairs. All checks passed. The execution record and source-hash input are embedded in `ps2_records` metadata. No external JSON file is needed to run the notebook.


**Project source:** [https://github.com/dku-comsci-econ206-Autumn2026/PS2-FP1-Yongzhuo-Wu](https://github.com/dku-comsci-econ206-Autumn2026/PS2-FP1-Yongzhuo-Wu) · release `ps2`  
**Demonstration:** [https://huggingface.co/spaces/dku-comsci-econ206-2026/AI-Sycophancy](https://huggingface.co/spaces/dku-comsci-econ206-2026/AI-Sycophancy)

## 1. Strategic environment
Two profit-maximizing apps A and B choose H (deliver a verifiable correction) or S (sycophancy) simultaneously.

| A \ B | H | S |
|---|---|---|
| H | (3,3) | (1,5) |
| S | (5,1) | (4,4) |

The stage is static and has complete payoff information. Simultaneous actions are unobserved when chosen, so complete information does not mean perfect information. Nash equilibrium is the appropriate normal-form concept. S strictly dominates without sales. This baseline is not the standard prisoner's-dilemma ranking: SS has greater joint baseline profit than HH.

Each H commits one identical correction slot before bidding. Fixed supply k is the number of H apps. Three unit-demand users bid private correction values, with quasilinear utility and no budget constraint. The platform calculation assumes the entire scenario value vector is known to the apps. Strong (5,4,3) and weak (3,2,1) are assumed values in common payoff units, not currency or estimates from the reflection.

One sealed round allocates the top k bids. Each winner pays bid k+1 and each H app receives one payment minus delivery cost c. Losers pay zero. Lower user ID resolves ties. There is no reserve, fee, or later round. k=0 means no trade. Verifiable delivery and payment enforcement are assumptions.



In [1]:
# Keep Colab's preloaded NumPy/SciPy untouched. Install only in our own venv.
import sys, subprocess, platform, json, tempfile, venv
from pathlib import Path

PINNED = {'nashpy':'0.0.43', 'numpy':'2.5.3', 'scipy':'1.18.1',
          'networkx':'3.7', 'deprecated':'3.0.0', 'wrapt':'2.5.0'}
if sys.version_info < (3, 12):
    raise RuntimeError('Use Python 3.12 or later for this fixed dependency set.')
ENV_DIR = Path(tempfile.gettempdir()) / f'ps2-fp1-solver-no-ensurepip-py{sys.version_info.major}{sys.version_info.minor}'
SOLVER_PYTHON = ENV_DIR / ('Scripts/python.exe' if sys.platform == 'win32' else 'bin/python')
if not (ENV_DIR / 'pyvenv.cfg').exists() or not SOLVER_PYTHON.exists():
    # Colab's Debian runtime may not provide ensurepip. Never invoke it.
    venv.EnvBuilder(with_pip=False).create(ENV_DIR)

PROBE = """
import json, platform, sys
from importlib.metadata import version
import numpy as np, scipy, nashpy, networkx, deprecated, wrapt
from numpy.testing import assert_allclose
assert_allclose(np.array([1., 2.]), [1., 2.])
assert np.__version__ == version('numpy')
print(json.dumps({'python':platform.python_version(),
                  'packages':{name:version(name) for name in %r},
                  'environment':sys.prefix}))
""" % list(PINNED)

def probe_solver():
    # -I ignores PYTHONPATH and user-site packages in the notebook environment.
    result = subprocess.run([str(SOLVER_PYTHON), '-I', '-c', PROBE],
                            text=True, capture_output=True)
    if result.returncode:
        return None
    return json.loads(result.stdout)

runtime_info = probe_solver()
if runtime_info is None or runtime_info['packages'] != PINNED:
    print('Preparing isolated Nashpy dependencies; the Colab environment is unchanged.', flush=True)
    # Host pip's --python installs only into the target, even without target pip.
    # This is supported since pip 22.3 and does not use ensurepip.
    subprocess.run([sys.executable, '-m', 'pip', '--python', str(SOLVER_PYTHON), 'install',
                    '--disable-pip-version-check', '--quiet', '--only-binary=:all:',
                    *[f'{name}=={wanted}' for name,wanted in PINNED.items()]],
                   check=True)
    runtime_info = probe_solver()
if runtime_info is None or runtime_info['packages'] != PINNED:
    raise RuntimeError('Isolated solver setup failed. Restart from a fresh runtime and run all cells.')

def installed(name):
    return runtime_info['packages'].get(name)

print('Notebook Python:', platform.python_version())
print('Solver Python:', runtime_info['python'])
print('Isolated package versions:', runtime_info['packages'])
print('PASS: imports including numpy.testing; no global NumPy/SciPy replacement.')


Preparing isolated Nashpy dependencies; the Colab environment is unchanged.
Notebook Python: 3.13.16
Solver Python: 3.13.16
Isolated package versions: {'nashpy': '0.0.43', 'numpy': '2.5.3', 'scipy': '1.18.1', 'networkx': '3.7', 'deprecated': '3.0.0', 'wrapt': '2.5.0'}
PASS: imports including numpy.testing; no global NumPy/SciPy replacement.


## 2. Algorithm and the three lenses
1. Count H apps, then rank sealed reports by descending bid and ascending user ID.
2. Allocate k slots, charge the highest losing bid and compute buyer utilities.
3. Add each H app's payment minus cost to its baseline payoff.
4. Check each unilateral action switch and retain mutual best responses.
5. Compare the no-service baseline, strong and weak demand, and cost 1. Verify matching payoff matrices independently with Nashpy.

Game theory predicts supply. Social choice evaluates correction surplus W = allocated value − ck and access k/3. Mechanism design changes allocation and payments through a fixed-supply k-Vickrey auction. Payments are transfers: buyer utility plus revenue equals allocated value. Revenue is not added again to W. Baseline profits, learning, harm from basic misleading answers and spillovers are outside W.


In [2]:
from itertools import product, combinations
from math import isclose, isfinite

BASE = {'HH': (3,3), 'HS': (1,5), 'SH': (5,1), 'SS': (4,4)}
DEMAND = {'strong': (5,4,3), 'weak': (3,2,1)}
COST = 0.0

def auction(values, k, bids=None, cost=0):
    values = tuple(values)
    bids = values if bids is None else tuple(bids)
    if not isinstance(k,int) or len(values)!=len(bids) or not 0<=k<len(values):
        raise ValueError('Require equal lengths and integer 0 <= k < number of bidders.')
    if any(not isfinite(x) or x<0 for x in values+bids+(cost,)):
        raise ValueError('Values, bids and cost must be finite and nonnegative.')
    order = sorted(range(len(bids)), key=lambda i: (-bids[i],i))
    winners = order[:k]
    price = bids[order[k]] if k else 0
    payments = tuple(price if i in winners else 0 for i in range(len(bids)))
    utilities = tuple((values[i] if i in winners else 0)-payments[i] for i in range(len(bids)))
    allocated = sum(values[i] for i in winners)
    return dict(winners=tuple(i+1 for i in winners),price=price,payments=payments,
                revenue=sum(payments),utilities=utilities,value=allocated,
                surplus=allocated-cost*k,access=k/len(values))

def game(values, cost=0):
    if len(values)!=3 or not isfinite(cost) or cost<0:
        raise ValueError('Require three users and finite nonnegative cost.')
    payoffs = {}
    for profile,base in BASE.items():
        price = auction(values,profile.count('H'),cost=cost)['price']
        payoffs[profile] = tuple(base[i]+(price-cost if profile[i]=='H' else 0) for i in range(2))
    return payoffs

def deviation_gain(payoffs, profile, player):
    switch = profile[:player]+('S' if profile[player]=='H' else 'H')+profile[player+1:]
    return payoffs[switch][player]-payoffs[profile][player]

def pure_ne(payoffs):
    return [p for p in BASE if all(deviation_gain(payoffs,p,i)<=1e-10 for i in range(2))]

print('Baseline matrix (A,B). Rows A=H,S. Columns B=H,S.')
print('        B=H       B=S')
print('A=H   (3,3)     (1,5)')
print('A=S   (5,1)     (4,4)')
print('Baseline pure NE:', pure_ne(BASE))


Baseline matrix (A,B). Rows A=H,S. Columns B=H,S.
        B=H       B=S
A=H   (3,3)     (1,5)
A=S   (5,1)     (4,4)
Baseline pure NE: ['SS']


## 3. Actual computed auction outcomes
Rows k=0,1,2 are fixed-supply outcomes. At zero cost, the strong-demand equilibrium realizes k=2 and the weak-demand equilibrium realizes k=0. Other rows are conditional counterfactuals, not observed transactions. User tuples follow ID order.


In [3]:
for name,values in DEMAND.items():
    print(f'\n{name.upper()} demand: {values}, cost={COST}')
    for k in range(3):
        a=auction(values,k,cost=COST)
        print(f"k={k}: winners={a['winners']}, price={a['price']}, revenue={a['revenue']}, "
              f"utilities={a['utilities']}, value={a['value']}, W={a['surplus']}, access={k}/3")
    payoffs=game(values,COST)
    print('Platform payoffs:', payoffs)
    print('Pure Nash equilibria:',pure_ne(payoffs))



STRONG demand: (5, 4, 3), cost=0.0
k=0: winners=(), price=0, revenue=0, utilities=(0, 0, 0), value=0, W=0.0, access=0/3
k=1: winners=(1,), price=4, revenue=4, utilities=(1, 0, 0), value=5, W=5.0, access=1/3
k=2: winners=(1, 2), price=3, revenue=6, utilities=(2, 1, 0), value=9, W=9.0, access=2/3
Platform payoffs: {'HH': (6.0, 6.0), 'HS': (5.0, 5), 'SH': (5, 5.0), 'SS': (4, 4)}
Pure Nash equilibria: ['HH']

WEAK demand: (3, 2, 1), cost=0.0
k=0: winners=(), price=0, revenue=0, utilities=(0, 0, 0), value=0, W=0.0, access=0/3
k=1: winners=(1,), price=2, revenue=2, utilities=(1, 0, 0), value=3, W=3.0, access=1/3
k=2: winners=(1, 2), price=1, revenue=2, utilities=(2, 1, 0), value=5, W=5.0, access=2/3
Platform payoffs: {'HH': (4.0, 4.0), 'HS': (3.0, 5), 'SH': (5, 3.0), 'SS': (4, 4)}
Pure Nash equilibria: ['SS']


## 4. Net-payment threshold and overturning condition
For sorted values v1 ≥ v2 ≥ v3, the H-minus-S profit gain is v3 − c − 2 against H, and v2 − c − 3 against S. Both must be positive for H to strictly dominate.

Strong values give 1−c against either action: HH is unique for 0 ≤ c < 1, SS is unique for c > 1. At c=1 both actions have equal payoff against either action of the other app. A's payoff rows coincide and B's payoff columns coincide. Every independent pair of mixed strategies is therefore a Nash equilibrium. Four pure profiles are only its corners.

Weak values give −1−c. S strictly dominates for every c ≥ 0. Extra revenue alone does not guarantee honesty. Neither the article nor this notebook estimates actual demand or delivery costs.


In [4]:
CASES=[('No service',None,0),('Strong c=0',DEMAND['strong'],0),
       ('Weak c=0',DEMAND['weak'],0),('Strong c=1',DEMAND['strong'],1)]
for label,values,cost in CASES:
    payoffs=BASE if values is None else game(values,cost)
    print(label, '| payoffs:', payoffs, '| pure NE:',pure_ne(payoffs))
    for profile in pure_ne(payoffs):
        print('  ',profile,'maximum profitable deviation:',max(0,*[deviation_gain(payoffs,profile,i) for i in range(2)]))
for cost in (0,0.5,1,2):
    print('Strong cost',cost,'| H-minus-S',1-cost,'| pure NE',pure_ne(game(DEMAND['strong'],cost)))


No service | payoffs: {'HH': (3, 3), 'HS': (1, 5), 'SH': (5, 1), 'SS': (4, 4)} | pure NE: ['SS']
   SS maximum profitable deviation: 0
Strong c=0 | payoffs: {'HH': (6, 6), 'HS': (5, 5), 'SH': (5, 5), 'SS': (4, 4)} | pure NE: ['HH']
   HH maximum profitable deviation: 0
Weak c=0 | payoffs: {'HH': (4, 4), 'HS': (3, 5), 'SH': (5, 3), 'SS': (4, 4)} | pure NE: ['SS']
   SS maximum profitable deviation: 0
Strong c=1 | payoffs: {'HH': (5, 5), 'HS': (4, 5), 'SH': (5, 4), 'SS': (4, 4)} | pure NE: ['HH', 'HS', 'SH', 'SS']
   HH maximum profitable deviation: 0
   HS maximum profitable deviation: 0
   SH maximum profitable deviation: 0
   SS maximum profitable deviation: 0
Strong cost 0 | H-minus-S 1 | pure NE ['HH']
Strong cost 0.5 | H-minus-S 0.5 | pure NE ['HH']
Strong cost 1 | H-minus-S 0 | pure NE ['HH', 'HS', 'SH', 'SS']
Strong cost 2 | H-minus-S -1 | pure NE ['SS']


## 5. Implementation checks and truthfulness
The finite check varies true values, both opponents' bids and own report on integers 0–6, for each bidder position and k=1,2. It checks 14,406 possible deviations. It tests implementation on a finite grid.

The general incentive argument fixes supply and opponents' reports. Their kth-highest bid sets the threshold. A user gains by winning only when its value covers that threshold. A truthful report obtains that outcome, with indifference possible at ties. This supplies the broader weak-dominance claim. General multi-unit demand, common-value uncertainty, unequal quality, endogenous post-bid supply, collusion or budget limits fall outside it. See Roughgarden (2014), CS364B Lecture 1, Section 3: https://theory.stanford.edu/~tim/w14/l/l21.pdf .


In [5]:
assert pure_ne(BASE)==['SS']
assert game(DEMAND['strong'])=={'HH':(6,6),'HS':(5,5),'SH':(5,5),'SS':(4,4)}
assert game(DEMAND['weak'])=={'HH':(4,4),'HS':(3,5),'SH':(5,3),'SS':(4,4)}
assert pure_ne(game(DEMAND['strong']))==['HH']
assert pure_ne(game(DEMAND['weak']))==['SS']
assert pure_ne(game(DEMAND['strong'],1))==list(BASE)
assert pure_ne(game(DEMAND['strong'],2))==['SS']
assert auction((5,4,3),0)['payments']==(0,0,0)
assert auction((3,3,3),2)['winners']==(1,2)
assert auction((5,4,3),1)['utilities']==(1,0,0)
assert auction((5,4,3),2)['utilities']==(2,1,0)
for values in DEMAND.values():
    for k in range(3):
        a=auction(values,k)
        assert min(a['utilities'])>=0
        assert a['value']==sum(sorted(values,reverse=True)[:k])
        assert isclose(sum(a['utilities'])+a['revenue'],a['value'])
        assert isclose(auction(values,k,cost=1)['surplus'],a['value']-k)
checks=0
for k,i in product((1,2),range(3)):
    for value,other1,other2 in product(range(7),repeat=3):
        bids=[other1,other2];bids.insert(i,value)
        truthful=auction(bids,k)['utilities'][i]
        for report in range(7):
            changed=bids.copy();changed[i]=report
            assert auction(bids,k,changed)['utilities'][i]<=truthful
            checks+=1
print(f'PASS: reference outcomes, accounting, ties, costs and {checks:,} finite bid-deviation checks.')


PASS: reference outcomes, accounting, ties, costs and 14,406 finite bid-deviation checks.


## 6. Matching independent Nashpy benchmark
Inputs below are hand-transcribed matrices from the article, not a generic example. Rows and columns follow H,S. Support enumeration should return SS, HH, SS, and the four pure cost-one profiles. Each returned profile is checked against best-response inequalities.

Nashpy may warn that the cost-one game is degenerate. A finite enumeration is not a listing of the continuous equilibrium set. The equal-row/column proof above establishes it. A 0.1 grid of 121 mixed pairs is an additional check, not a completeness proof. Nashpy documentation: https://nashpy.readthedocs.io/en/stable/how-to/handle-degenerate-games.html .

The package check runs in a dedicated virtual environment and fresh subprocess, isolated from Colab's loaded scientific stack. It receives the notebook-generated payoffs and verifies them against independently transcribed matrices.


In [6]:
# Independent fixed matrices run in a fresh process inside the isolated venv.
SOLVER_CODE = "import json, sys, warnings\nfrom itertools import product\nimport numpy as np\nimport nashpy as nash\ncomputed=json.load(sys.stdin)\nMANUAL=[('No service',[[3,1],[5,4]],[[3,5],[1,4]],['SS']),\n        ('Strong c=0',[[6,5],[5,4]],[[6,5],[5,4]],['HH']),\n        ('Weak c=0',[[4,3],[5,4]],[[4,5],[3,4]],['SS']),\n        ('Strong c=1',[[5,4],[5,4]],[[5,5],[4,4]],['HH','HS','SH','SS'])]\ndef residual(A,B,x,y):\n    ua=A@y;ub=x@B\n    return max(0,float(np.max(ua)-x@ua),float(np.max(ub)-ub@y))\nreceipts=[]\nfor label,a,b,expected in MANUAL:\n    A=np.array(a,dtype=float);B=np.array(b,dtype=float)\n    with warnings.catch_warnings(record=True) as captured:\n        warnings.simplefilter('always')\n        solved=list(nash.Game(A,B).support_enumeration())\n    pure=[];maximum=0\n    for x,y in solved:\n        gap=residual(A,B,x,y);assert gap<=1e-10;maximum=max(maximum,gap)\n        if np.isclose(x.max(),1) and np.isclose(y.max(),1):\n            pure.append('HS'[int(x.argmax())]+'HS'[int(y.argmax())])\n    assert set(pure)==set(expected)\n    mapped={h+k:[float(A[i,j]),float(B[i,j])] for i,h in enumerate('HS') for j,k in enumerate('HS')}\n    assert mapped==computed[label]\n    receipts.append(dict(case=label,A=A.tolist(),B=B.tolist(),pure_NE=pure,\n                         max_deviation=maximum,warnings=[str(w.message) for w in captured]))\nA=np.array(MANUAL[-1][1],float);B=np.array(MANUAL[-1][2],float)\nassert np.array_equal(A[0],A[1]) and np.array_equal(B[:,0],B[:,1])\nfor p,q in product(np.linspace(0,1,11),repeat=2):\n    assert residual(A,B,np.array([p,1-p]),np.array([q,1-q]))<=1e-10\nprint(json.dumps({'receipts':receipts,'mixed_pairs_checked':121}))\n"
computed = {label: BASE if values is None else game(values,cost)
            for label,values,cost in CASES}
result = subprocess.run([str(SOLVER_PYTHON), '-I', '-c', SOLVER_CODE],
                        input=json.dumps(computed), text=True, capture_output=True)
if result.returncode:
    raise RuntimeError('Independent Nashpy check failed:\n' + result.stderr)
solver_result = json.loads(result.stdout)
solver_receipts = solver_result['receipts']
assert solver_result['mixed_pairs_checked']==121
for record in solver_receipts:
    print(record['case'],'| Nashpy pure NE:',record['pure_NE'],
          '| maximum deviation:',record['max_deviation'])
    for message in record['warnings']:
        print('  Degeneracy caveat:',message.strip())
print('PASS: four matching Nashpy cases, deviation residuals and 121 cost-boundary mixed pairs.')
print('Continuum conclusion uses equal payoff rows/columns, not finite enumeration.')


No service | Nashpy pure NE: ['SS'] | maximum deviation: 0
Strong c=0 | Nashpy pure NE: ['HH'] | maximum deviation: 0
Weak c=0 | Nashpy pure NE: ['SS'] | maximum deviation: 0
Strong c=1 | Nashpy pure NE: ['HH', 'HS', 'SH', 'SS'] | maximum deviation: 0
  Degeneracy caveat: An even number of (4) equilibria was returned. This
indicates that the game is degenerate. Consider using another algorithm
to investigate.
PASS: four matching Nashpy cases, deviation residuals and 121 cost-boundary mixed pairs.
Continuum conclusion uses equal payoff rows/columns, not finite enumeration.


## 7. Social-choice comparison at fixed capacity
Compare auction allocation with an equal lottery over the three possible winner pairs at k=2. All users have access probability 2/3 under the lottery. Under truthful auction allocation the lowest-value user has probability 0. Subtract the same 2c from both gross values for a cost comparison.

The lottery has no proposed payment or financing rule here. Its allocation result cannot be substituted into the platform game while retaining the auction's honesty prediction. Payments and welfare remain distinct.

A separate synthetic voting illustration has users 1 and 2 prefer an accuracy-first service goal and user 3 prefer agreement-first. One-person-one-vote selects accuracy 2–1 independent of wealth. It selects a service goal, does not determine whether 17×6=102, and leaves a minority preference. No user votes were collected.


In [7]:
for name,values in DEMAND.items():
    pairs=list(combinations(range(3),2))
    lottery_value=sum(sum(values[i] for i in pair) for pair in pairs)/len(pairs)
    probabilities=[sum(i in pair for pair in pairs)/len(pairs) for i in range(3)]
    allocated=auction(values,2)['value']
    assert isclose(lottery_value,8 if name=='strong' else 4)
    assert all(isclose(p,2/3) for p in probabilities)
    assert isclose(allocated-lottery_value,1)
    print(name,'| auction allocated value:',allocated,'| equal lottery expected value:',lottery_value,
          '| lottery access:',probabilities,'| auction minimum access: 0')
    print('  At c=1: auction W=',allocated-2,'lottery expected W=',lottery_value-2)
print('PASS: exact fixed-capacity allocation comparison. Lottery financing/equilibrium unspecified.')


strong | auction allocated value: 9 | equal lottery expected value: 8.0 | lottery access: [0.6666666666666666, 0.6666666666666666, 0.6666666666666666] | auction minimum access: 0
  At c=1: auction W= 7 lottery expected W= 6.0
weak | auction allocated value: 5 | equal lottery expected value: 4.0 | lottery access: [0.6666666666666666, 0.6666666666666666, 0.6666666666666666] | auction minimum access: 0
  At c=1: auction W= 3 lottery expected W= 2.0
PASS: exact fixed-capacity allocation comparison. Lottery financing/equilibrium unspecified.


## 8. Retained author evidence and behavioral boundary
**Observed n=1:** the author's October 5 self-reflection on the false claim 17×6=112. A corrects to 102, B falsely affirms 112. The record selects A, states WTP 3 for A and 0 for B, and gives “I prefer honest answers.” The selected strong-demand field is an interface setting. These entries do not identify the three-user vector, record a purchase or support a population rate.

```json
{
  "date": "2026-10-05T14:20:54.948Z",
  "demand": "strong",
  "activity": "Self-reflection",
  "preference": "A",
  "wtpA": "3",
  "wtpB": "0",
  "reason": "I prefer honest answers."
}
```

An accuracy mechanism predicts correction choice and delayed factual recall. Project cues, an obvious arithmetic error, and liking A's wording are alternative explanations. A discriminating design would match tone, randomize display order across multiple checkable items, measure delayed accuracy separately from liking, and compare stated payment with an incentivized purchase. **No participant study is scheduled.** Interface test exports do not become additional evidence.

In the tutoring case, educators would assess learning, providers would disclose cost and delivery, public institutions could support basic correction and independent reviewers would audit and hear appeals. Minority preferences and user control matter. Free basic correction, subsidies, a correction log and independent human appeal are proposals, not implemented protections. SDG 4 learning and equal-access effects remain unevaluated.

## 9. Studio development and verified literature
The author's five Studio screenshots record baseline SS, strong HH, weak SS, and all four pure cost-one profiles. Studio's Cooperate/Defect labels map to H/S here. The article includes an English translation of the author's reflection; the original screenshots are in `supporting-materials/studio/`. Studio checks the matrices; it does not measure firms or customers. Course Studio: https://gt-tools-demos-sigma.vercel.app/ .

Sharma et al. (ICLR 2024) evaluate assistant tasks and preference evidence. Cheng et al. (Science 2026) study social affirmation in three preregistered experiments (N=2,405). Neither estimates correction-slot demand or delivery costs. Peer prediction already rewards informative reports and costly effort: [Miller et al. (2005)](https://doi.org/10.1287/mnsc.1050.0379) and [Witkowski et al. (2013)](https://doi.org/10.1609/hcomp.v1i1.13089). Here verified delivery is assumed, and customer auctions fund competing apps. The contribution is a bounded application linking supply, net payment and access, without a priority claim.

Sources and software: [Sharma et al.](https://openreview.net/forum?id=tvhaxkMKAn), [Cheng et al.](https://doi.org/10.1126/science.aec8352), [Nash 1950](https://doi.org/10.1073/pnas.36.1.48), [Vickrey 1961](https://doi.org/10.1111/j.1540-6261.1961.tb02789.x), and Nashpy's official documentation cited above. ACM numbering and full references remain in the article.

## 10. Execution and assistance
The final cell hashes the actual source of `auction`, `game`, `deviation_gain` and `pure_ne`, the matching Nashpy subprocess source, and the model parameters. SHA-256 uses UTF-8 canonical JSON with sorted keys and compact separators. Cell formatting, timestamps, package versions and the test harness are outside this declared scope. Notebook metadata `ps2_records.model_source` stores the saved hash input so another reader can recompute it; cell 8 also exports a runtime copy as `model-source.json`. Changing a covered function or parameter changes the digest.

I ran the final notebook in Colab on October 6, 2026. Its saved outputs include the 121 mixed-pair check and the source digest computed by the final cell. Metadata `ps2_records.author_colab_run` retains that execution record, `author_colab_original_json` retains its original JSON text, and `author_identity` records my confirmation of the run. The saved outputs, execution counts and code are unchanged from my Colab download. Each rerun prints its own time, versions and check results. Computation and interface tests add no human observations: bids, purchases, platform choices and delayed learning each have observed n=0.

| Author's Colab execution | Value |
|---|---|
| UTC time | 2026-10-06 11:08:10 |
| Beijing time | 2026-10-06 19:08:10 |
| Python / solver Python | 3.13.16 / 3.13.16 |
| Code cells | 8 |
| Finite bid-deviation checks | 14,406 |
| Matching Nashpy cases | 4 |
| Cost-boundary mixed pairs | 121 |
| Result | ALL CHECKS PASS |


In [8]:
from datetime import datetime, timezone
from pathlib import Path
import hashlib, inspect, json

parameters = {'baseline': BASE, 'demand': DEMAND, 'baseline_cost': COST,
              'cost_challenge': 1}
model_source = {
    'scope': 'four model functions, matching Nashpy subprocess source, and parameters',
    'functions': {fn.__name__: inspect.getsource(fn).strip()
                  for fn in (auction, game, deviation_gain, pure_ne)},
    'solver_source': SOLVER_CODE,
    'parameters': parameters,
}
canonical_source = json.dumps(model_source, sort_keys=True, separators=(',', ':'),
                              ensure_ascii=False).encode('utf-8')
source_digest = hashlib.sha256(canonical_source).hexdigest()
run_receipt = {
    'project': 'PS2-FP1',
    'release': 'ps2',
    'operator': globals().get('RUN_OPERATOR', 'Notebook runner; identity documented separately'),
    'executed_utc': datetime.now(timezone.utc).isoformat(timespec='seconds'),
    'python': platform.python_version(),
    'solver_python': runtime_info['python'],
    'solver_environment': 'dedicated venv; fresh isolated subprocess',
    'packages': {name: installed(name) for name in PINNED},
    'model_code_sha256': source_digest,
    'hash_scope': model_source['scope'],
    'hash_encoding': 'SHA-256 of UTF-8 JSON; sorted keys; compact separators',
    'parameters': parameters,
    'finite_bid_deviations': checks,
    'mixed_pairs_checked': solver_result['mixed_pairs_checked'],
    'nashpy_cases': solver_receipts,
    'randomness': 'none',
    'behavioral_evidence': 'one author reflection; observed bids, purchases, app actions and learning n=0',
    'status': 'PASS'
}
Path('model-source.json').write_text(json.dumps(model_source, indent=2, ensure_ascii=False) + '\n')
Path('run-record.json').write_text(json.dumps(run_receipt, indent=2, ensure_ascii=False) + '\n')
print(json.dumps(run_receipt, indent=2))
print('Saved model-source.json and run-record.json. Hash scope:', model_source['scope'])
print('ALL CHECKS PASS.')


{
  "project": "PS2-FP1",
  "release": "ps2",
  "operator": "Notebook runner; identity documented separately",
  "executed_utc": "2026-10-06T11:08:10+00:00",
  "python": "3.13.16",
  "solver_python": "3.13.16",
  "solver_environment": "dedicated venv; fresh isolated subprocess",
  "packages": {
    "nashpy": "0.0.43",
    "numpy": "2.5.3",
    "scipy": "1.18.1",
    "networkx": "3.7",
    "deprecated": "3.0.0",
    "wrapt": "2.5.0"
  },
  "model_code_sha256": "dafd872d35ebc327d41317427cb3c61bca80eba9bfea66e45e0736e3ceecb15d",
  "hash_scope": "four model functions, matching Nashpy subprocess source, and parameters",
  "hash_encoding": "SHA-256 of UTF-8 JSON; sorted keys; compact separators",
  "parameters": {
    "baseline": {
      "HH": [
        3,
        3
      ],
      "HS": [
        1,
        5
      ],
      "SH": [
        5,
        1
      ],
      "SS": [
        4,
        4
      ]
    },
    "demand": {
      "strong": [
        5,
        4,
        3
      ],
      "